# Denoising Diffusion Restoration Models (DDRM) — Implementation / 구현

**Paper**: Kawar, B., Elad, M., Ermon, S., & Song, J. (2022). *Proc. NeurIPS*. [arXiv:2201.11793]

## Overview / 개요

이 노트북은 DDRM의 두 핵심 아이디어를 작은 toy 영상 (16×16)에서 시연한다:
1. **SVD spectral decomposition**: 단순한 4× block-averaging downsampling matrix $H$의 SVD를 명시적으로 계산.
2. **Spectral-space diffusion sampling**: 각 singular component별로 세 case (Eq. 5)를 적용하여 inverse problem을 풀이. 사전학습 DDPM 대신 *Gaussian-smoothing 디노이저*를 stub으로 사용 — 알고리즘의 mechanics을 보이는 데 충분.

This notebook demonstrates DDRM's two core ideas on a tiny 16×16 image:
1. **Explicit SVD** of a 4× block-averaging $H$ matrix.
2. **Spectral-space DDRM sampling** with the three-case branch (Eq. 5). We use a Gaussian-smoothing denoiser stub instead of a real DDPM — the algorithm's mechanics are the same.

We do NOT need GPU or a pretrained DDPM.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray
from scipy.ndimage import gaussian_filter

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 10
rng = np.random.default_rng(42)
print('Modules loaded.')

---

## Part 1: Toy image and degradation matrix H / 토이 영상과 분해 행렬 H

16×16 piecewise-smooth image, 4× block-averaging downsampling → 4×4 measurement.

$H$는 16²×4² = 256×16 block-averaging matrix. 각 행은 하나의 4×4 input block의 mean을 계산.

We make $H$ explicit so we can SVD it (next part) and apply DDRM's spectral-space sampling.

In [ ]:
def make_smooth_image(size: int = 16) -> NDArray[np.float64]:
    """Synthetic smooth image with low-rank structure."""
    yy, xx = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing='ij')
    img = 0.6 * np.exp(-(xx ** 2 + yy ** 2) / 0.4 ** 2)
    img += 0.4 * np.exp(-((xx + 0.5) ** 2 + (yy - 0.3) ** 2) / 0.2 ** 2)
    return np.clip(img, 0.0, 1.0)


def build_block_avg_H(size: int, factor: int) -> NDArray[np.float64]:
    """Build block-averaging matrix H of shape (m=size//factor)^2 x size^2.

    Each row averages a non-overlapping factor x factor block.
    """
    n = size * size
    m_lin = size // factor
    m = m_lin * m_lin
    H = np.zeros((m, n))
    block_size = factor * factor
    for bi in range(m_lin):
        for bj in range(m_lin):
            row_idx = bi * m_lin + bj
            for i in range(factor):
                for j in range(factor):
                    pixel_row = bi * factor + i
                    pixel_col = bj * factor + j
                    pixel_idx = pixel_row * size + pixel_col
                    H[row_idx, pixel_idx] = 1.0 / block_size
    return H


SIZE = 16
FACTOR = 4
clean_img = make_smooth_image(SIZE)
x_flat = clean_img.flatten()
H = build_block_avg_H(SIZE, FACTOR)
print(f'Image shape: {clean_img.shape}, flattened length n = {x_flat.shape[0]}')
print(f'H shape: {H.shape} (m={H.shape[0]}, n={H.shape[1]})')

# Apply degradation (with optional measurement noise)
sigma_y = 0.02
y_noiseless = H @ x_flat
y_obs = y_noiseless + sigma_y * rng.standard_normal(y_noiseless.shape)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
axes[0].imshow(clean_img, cmap='gray', vmin=0, vmax=1); axes[0].set_title(f'Original ({SIZE}×{SIZE})')
axes[1].imshow(y_obs.reshape(4, 4), cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Measurement (4×4, $\\sigma_y$={sigma_y})')
# bicubic baseline
from scipy.ndimage import zoom
baseline = zoom(y_obs.reshape(4, 4), zoom=4, order=3)
axes[2].imshow(baseline, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Bicubic upsample baseline')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 2: SVD of degradation matrix / 분해 행렬의 SVD

$H = U \Sigma V^\top$, $U \in \mathbb R^{16\times 16}$, $V \in \mathbb R^{256\times 256}$, $\Sigma$ rectangular diagonal $16\times 256$.

The 16 non-zero singular values reveal *which* spectral components of $x$ are measured by $y$. The 240 zero singular values correspond to spectral components in $\ker(H)$ — pure prior generation in DDRM.

In [ ]:
U, s_values, Vt = np.linalg.svd(H, full_matrices=True)
V = Vt.T
print(f'U shape: {U.shape}, V shape: {V.shape}, # singular values: {s_values.size}')
print(f'Singular value range: [{s_values.min():.4f}, {s_values.max():.4f}]')
print(f'For 4x4 block averaging on 16x16 image, expected s_i = 1/4 = 0.25 for all 16 components.')
print(f"Number of nonzero s_i: {(s_values > 1e-8).sum()} / 16 (rest of V columns span ker(H))")

# Visualise first few V columns reshaped to image (these are the 16 'measurable' modes)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(16):
    ax = axes[i // 8, i % 8]
    mode = V[:, i].reshape(SIZE, SIZE)
    ax.imshow(mode, cmap='RdBu', vmin=-0.1, vmax=0.1)
    ax.set_title(f's_{i}={s_values[i]:.3f}', fontsize=8)
    ax.axis('off')
plt.suptitle('First 16 V columns reshaped (= measurable spectral modes)')
plt.tight_layout(); plt.show()

print("\nThe first 16 V-columns are the 'identity-block' indicators of the 4x4 grid — modes the")
print("measurement directly probes. The remaining 240 V-columns span ker(H) — spectral modes the")
print("measurement is BLIND to. DDRM samples these 240 components from the diffusion prior.")

---

## Part 3: Spectral-space DDRM sampling (toy version) / Spectral-space DDRM 샘플링 (토이 버전)

Implement the three-case update (Eq. 5) at each step. For the denoiser, we use Gaussian smoothing as a stub — a real DDRM uses a pre-trained DDPM (e.g., Dhariwal-Nichol on ImageNet).

Eq. 5 (case $s_i = 0$ for the unmeasured 240 spectral components, case 3 for the 16 measured components since $\sigma_T \gg \sigma_y/s_i$):
- $s_i = 0$: $\bar x_t^{(i)} \sim \mathcal N(\bar x_0^{(i)} + \sqrt{1-\eta^2}\sigma_t \frac{\bar x_{t+1}^{(i)} - \bar x_0^{(i)}}{\sigma_{t+1}}, \eta^2 \sigma_t^2)$
- $s_i > 0, \sigma_t \ge \sigma_y/s_i$: $\bar x_t^{(i)} \sim \mathcal N((1-\eta_b)\bar x_0^{(i)} + \eta_b \bar y^{(i)}, \sigma_t^2 - \sigma_y^2/s_i^2 \cdot \eta_b^2)$

In [ ]:
def smoothing_denoiser(x_flat: NDArray[np.float64], sigma_t: float, size: int) -> NDArray[np.float64]:
    """Stub for DDPM denoiser: returns predicted x_0 from x_t at noise level sigma_t.

    Real DDRM uses a pre-trained DDPM. Here we approximate with Gaussian smoothing of bandwidth
    proportional to sigma_t (less smoothing as sigma_t -> 0).
    """
    img = x_flat.reshape(size, size)
    bandwidth = max(0.5 + 2.0 * sigma_t, 0.5)
    smoothed = gaussian_filter(img, sigma=bandwidth, mode='reflect')
    return smoothed.flatten()


def ddrm_sample(
    H: NDArray[np.float64],
    y_obs: NDArray[np.float64],
    sigma_y: float,
    denoiser_fn,
    size: int,
    T: int = 20,
    sigma_T: float = 1.0,
    sigma_0: float = 0.01,
    eta: float = 0.85,
    eta_b: float = 1.0,
    seed: int = 0,
) -> NDArray[np.float64]:
    """Toy DDRM sampler in spectral space.

    Args:
        H: degradation matrix (m, n).
        y_obs: observed measurements (m,).
        sigma_y: measurement noise std.
        denoiser_fn: callable (x_flat, sigma_t) -> predicted x_0.
        size: image side (n = size*size).
        T: number of diffusion steps.
        sigma_T: largest noise level (start).
        sigma_0: smallest noise level (terminal).
        eta: stochastic-vs-deterministic mix (DDIM-style).
        eta_b: measurement injection coefficient (Theorem 3.2 uses eta_b = 2 sigma_t^2/(sigma_t^2 + sigma_y^2/s_i^2)).
        seed: RNG seed.

    Returns:
        Sampled x_0 (n,).
    """
    g = np.random.default_rng(seed)
    U, s_values, Vt = np.linalg.svd(H, full_matrices=True)
    V = Vt.T
    n = H.shape[1]
    m = H.shape[0]

    # Pseudo-inverse of Sigma to compute bar_y.
    s_inv = np.where(s_values > 1e-8, 1.0 / s_values, 0.0)
    # bar_y_full has length n (zeros for indices >= m): bar_y[i] = (Sigma^+ U^T y)_i
    bar_y = np.zeros(n)
    bar_y[:m] = s_inv * (U.T @ y_obs)

    # Effective noise std per spectral component
    sigma_eff = np.zeros(n)
    sigma_eff[:m] = sigma_y * s_inv  # for s_i > 0
    sigma_eff[m:] = np.inf  # placeholder for s_i = 0 components

    # Schedule (linear sigma_t from sigma_T to sigma_0)
    sigma_schedule = np.linspace(sigma_T, sigma_0, T + 1)

    # Initialise bar_x_T from q^(T)(.|x_0, y) (Eq. 4).
    bar_x = np.zeros(n)
    for i in range(n):
        if i < m and s_values[i] > 1e-8:
            var = max(sigma_T ** 2 - sigma_eff[i] ** 2, 1e-6)
            bar_x[i] = bar_y[i] + np.sqrt(var) * g.standard_normal()
        else:
            bar_x[i] = sigma_T * g.standard_normal()

    # Reverse chain
    for t in range(T, 0, -1):
        sigma_t = sigma_schedule[t]
        sigma_t_next = sigma_schedule[t - 1]
        x_flat = V @ bar_x
        x_pred = denoiser_fn(x_flat, sigma_t)  # predicted x_0
        bar_x_pred = V.T @ x_pred

        new_bar_x = np.zeros(n)
        for i in range(n):
            s_i = s_values[i] if i < m else 0.0
            if s_i < 1e-8:
                # Case s_i = 0 (Eq. 5 first branch)
                mean = bar_x_pred[i] + np.sqrt(1 - eta ** 2) * sigma_t_next * (
                    (bar_x[i] - bar_x_pred[i]) / max(sigma_t, 1e-6)
                )
                var = (eta * sigma_t_next) ** 2
            else:
                sig_y_over_s = sigma_eff[i]
                if sigma_t_next >= sig_y_over_s:
                    # Case 3: diffusion noisier than measurement
                    mean = (1 - eta_b) * bar_x_pred[i] + eta_b * bar_y[i]
                    var = max(sigma_t_next ** 2 - sig_y_over_s ** 2 * eta_b ** 2, 1e-8)
                else:
                    # Case 2: measurement noisier than diffusion (rare in our tiny example)
                    mean = bar_x_pred[i] + np.sqrt(1 - eta ** 2) * sigma_t_next * (
                        (bar_y[i] - bar_x_pred[i]) / max(sig_y_over_s, 1e-6)
                    )
                    var = (eta * sigma_t_next) ** 2
            new_bar_x[i] = mean + np.sqrt(var) * g.standard_normal()
        bar_x = new_bar_x

    # Final: convert back to pixel space
    return V @ bar_x


x_recon = ddrm_sample(
    H, y_obs, sigma_y,
    denoiser_fn=lambda x, s: smoothing_denoiser(x, s, SIZE),
    size=SIZE, T=20, sigma_T=0.8, sigma_0=0.02, eta=0.85, eta_b=1.0, seed=7
)
img_recon = x_recon.reshape(SIZE, SIZE)

def psnr(a, b):
    mse = np.mean((a - b) ** 2)
    return float('inf') if mse == 0 else 10 * np.log10(1.0 / max(mse, 1e-12))

print(f'Bicubic baseline PSNR: {psnr(baseline, clean_img):.2f} dB')
print(f'DDRM (toy) PSNR     : {psnr(img_recon, clean_img):.2f} dB')

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(clean_img, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Original')
axes[1].imshow(y_obs.reshape(4, 4), cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Measurement (4×4, σ_y={sigma_y})')
axes[2].imshow(baseline, cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Bicubic\nPSNR={psnr(baseline, clean_img):.2f}dB')
axes[3].imshow(img_recon, cmap='gray', vmin=0, vmax=1)
axes[3].set_title(f'DDRM toy (T=20)\nPSNR={psnr(img_recon, clean_img):.2f}dB')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 4: Inspect the spectral-component update / Spectral component 업데이트 검사

DDRM의 핵심: 각 spectral component가 *독립*적으로 update됨. 측정된 16개 component는 Case 3 (measurement injection), 나머지 240개는 Case 1 (pure prior). 이를 시각화.

In [ ]:
# Compute bar_y for the 16 measured components
U, s_values, Vt = np.linalg.svd(H, full_matrices=True)
V = Vt.T
s_inv = np.where(s_values > 1e-8, 1.0 / s_values, 0.0)
bar_y = s_inv * (U.T @ y_obs)
bar_x_truth = V.T @ x_flat
bar_x_recon = V.T @ x_recon

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.scatter(np.arange(16), bar_x_truth[:16], color='black', s=40, label='ground truth $\\bar x_0^{(i)}$')
ax.scatter(np.arange(16), bar_y, color='red', s=20, label='measurement $\\bar y^{(i)}$')
ax.scatter(np.arange(16), bar_x_recon[:16], color='blue', s=20, marker='x', label='DDRM recon')
ax.set_xlabel('spectral index $i$ (0..15, $s_i > 0$ → measured)')
ax.set_ylabel('value')
ax.set_title('Measured spectral components: case 3 update')
ax.legend()

ax = axes[1]
indices_unmeasured = np.arange(16, 16 + 50)
ax.scatter(indices_unmeasured, bar_x_truth[indices_unmeasured], color='black', s=20, label='truth')
ax.scatter(indices_unmeasured, bar_x_recon[indices_unmeasured], color='blue', s=10, marker='x', label='DDRM')
ax.set_xlabel('spectral index $i$ (16..65, $s_i = 0$ → unmeasured)')
ax.set_ylabel('value')
ax.set_title('Unmeasured spectral components: case 1 update (pure prior)')
ax.legend()
plt.tight_layout(); plt.show()

# Quantitative
err_measured = np.abs(bar_x_recon[:16] - bar_x_truth[:16]).mean()
err_unmeasured = np.abs(bar_x_recon[16:] - bar_x_truth[16:]).mean()
print(f'Mean abs error on measured 16 spectral components  : {err_measured:.4f}')
print(f'Mean abs error on unmeasured 240 spectral components: {err_unmeasured:.4f}')
print('Measured components are tight (constrained by measurement); unmeasured are free, sampled from prior.')

---

## Part 5: Multi-sample diversity / 다중 샘플 다양성

Different RNG seeds → different $x_0$ samples consistent with the *same* measurement. This is the posterior-sampling property of DDRM.

다른 seed에 대해 DDRM이 *같은 측정값에 일관*되면서도 다양한 복원 결과를 생성하는지 확인.

In [ ]:
n_samples = 4
samples = []
for seed in range(n_samples):
    s = ddrm_sample(
        H, y_obs, sigma_y,
        denoiser_fn=lambda x, sigma_t: smoothing_denoiser(x, sigma_t, SIZE),
        size=SIZE, T=20, sigma_T=0.8, sigma_0=0.02, eta=0.85, eta_b=1.0, seed=200 + seed
    )
    samples.append(s.reshape(SIZE, SIZE))

fig, axes = plt.subplots(1, n_samples + 2, figsize=(15, 3))
axes[0].imshow(clean_img, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Truth')
axes[0].axis('off')
for i, s in enumerate(samples):
    axes[i + 1].imshow(s, cmap='gray', vmin=0, vmax=1)
    axes[i + 1].set_title(f'sample {i}\nPSNR={psnr(s, clean_img):.1f}dB'); axes[i + 1].axis('off')
axes[-1].imshow(np.mean(samples, axis=0), cmap='gray', vmin=0, vmax=1)
axes[-1].set_title(f'mean\nPSNR={psnr(np.mean(samples, axis=0), clean_img):.1f}dB')
axes[-1].axis('off')
plt.tight_layout(); plt.show()

# Verify all samples are consistent with measurement
consistency = []
for s in samples:
    y_re = H @ s.flatten()
    consistency.append(np.abs(y_re - y_obs).max())
print(f'Max |H x_recon - y_obs| across {n_samples} samples: {max(consistency):.4f}')
print(f'(Should be small — DDRM samples are constrained by the measurement.)')

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| SVD of degradation $H$ | §3.2 Eq. 3 | Part 1, 2 — explicit np.linalg.svd | DDNM, $\Pi$GDM (use null-space projection variants) |
| Three-case spectral update | §3.2 Eq. 5 | Part 3 — explicit case branching | DPS (single-case approximate gradient) |
| Pretrained DDPM as prior | §3.3 Theorem 3.2 | Part 3 — Gaussian-smoothing stub | Imagen, Stable Diffusion (latent DDPMs) |
| Multi-sample posterior diversity | §5.3 Fig. 5 | Part 5 — 4 random-seed samples | Posterior sampling for ill-posed inverse |
| Spectral measured vs unmeasured | §3.2 Eq. 4–5 cases | Part 4 — explicit visualization | Null-space decomposition |

### Why we used a stub denoiser / 왜 stub denoiser를 썼는가

원 논문은 ImageNet, CelebA-HQ, LSUN bedroom/cats에 학습된 거대 DDPM (Dhariwal-Nichol 2021) 가중치를 사용한다. 이 노트북의 목적은 *알고리즘의 mechanics*을 보이는 것 — Gaussian smoothing이 형편없는 사전이지만 (자연 영상 detail 생성 못함) **spectral-space update의 흐름은 정확히 동일**.

The original paper uses pretrained DDPMs from Dhariwal-Nichol (2021). We use a Gaussian-smoothing stub — an obviously inferior prior, but the algorithmic mechanics are identical. To get state-of-the-art DDRM PSNRs (~26.55 dB on ImageNet 4× SR), one plugs in the actual DDPM weights and runs the same code with $T = 20$ uniformly-sampled timesteps.

### Connections to other papers / 다른 논문과의 연결

- **Paper #25 (Kadkhodaie-Simoncelli)**: DDRM is the DDPM-scale evolution. Paper #25's Algorithm 2 is a *single-step* analogue of DDRM's spectral chain.
- **Paper #27 (Cold Diffusion)**: shows that the noise-centric framework here is *one* of many — same restoration tasks (deblurring, inpainting, SR) can be solved with deterministic forward chains.
- **Paper #16 (Noise2Noise)**: provides the means to train the DDPM denoiser without clean data — combine for fully self-supervised DDRM.
- **Paper #1 (Donoho-Johnstone)**: the historical predecessor of "sparse-spectral-prior" inverse problem solving — DDRM's V-decomposition is the modern operator-theoretic generalization.

### Take-away / 결론

본 노트북은 DDRM의 두 핵심을 작은 16×16 영상에서 명시적으로 시연:
1. **SVD가 inverse problem을 16+240 = 256개의 *독립 1-D 문제*로 분해한다** (Part 2 시각화).
2. **3-case spectral update가 measurement constraint와 prior를 무리 없이 결합한다** (Part 4의 measured/unmeasured 분리).
3. **Multi-sample은 모두 같은 측정값에 consistent하면서 다양한 복원을 제공한다** (Part 5).

이 toy 알고리즘에 *진짜 DDPM*을 끼워넣으면 ImageNet 4× SR에서 26.55 dB를 달성한다 (paper Table 1) — 동일 코드, 다른 디노이저 가중치.

This notebook explicitly demonstrates DDRM's two ingredients on a 16×16 image: (1) SVD decomposes the inverse problem into 16 measured + 240 unmeasured 1-D problems; (2) the three-case spectral update branches on $s_i$ and $\sigma_t$ vs $\sigma_y/s_i$. Replacing the Gaussian-smoothing stub with a pretrained DDPM gives the paper's headline 26.55 dB on ImageNet 4× SR — same code, different weights.